In [ ]:
!pip install -q timm scikit-learn pandas


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import timm

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
root_dir = "/workspace/dataset"
splits = ["train", "val", "test"]

total_removed = 0

for split in splits:
    split_path = os.path.join(root_dir, split)
    print(f"\nCleaning {split.upper()}...")

    removed = 0
    before = 0

    for class_folder in os.listdir(split_path):
        class_path = os.path.join(split_path, class_folder)
        if not os.path.isdir(class_path):
            continue

        for file in os.listdir(class_path):
            file_path = os.path.join(class_path, file)
            before += 1

            # Remove hidden
            if file.startswith("."):
                os.remove(file_path)
                removed += 1
                continue

            # Remove empty
            if os.path.getsize(file_path) == 0:
                os.remove(file_path)
                removed += 1
                continue

            # Remove corrupted
            try:
                with Image.open(file_path) as img:
                    img.verify()
            except:
                os.remove(file_path)
                removed += 1

    total_removed += removed
    print(f"Files removed in {split}: {removed}")

print("\nTotal removed:", total_removed)
print("Dataset fully cleaned ✅")


Cleaning TRAIN...
Files removed in train: 0

Cleaning VAL...
Files removed in val: 0

Cleaning TEST...
Files removed in test: 0

Total removed: 0
Dataset fully cleaned ✅


In [ ]:
class CustomGroupedDataset(ImageFolder):
    def __init__(self, root, transform=None):
        super().__init__(root, transform=transform)

        new_samples = []
        new_targets = []

        for path, target in self.samples:
            folder_name = path.split(os.sep)[-2]
            class_id = int(folder_name.split(".")[0])

            if 1 <= class_id <= 3:
                new_label = class_id - 1
            elif 5 <= class_id <= 18:
                new_label = class_id - 2
            elif class_id == 21:
                new_label = 17
            elif 22 <= class_id <= 31:
                new_label = 18
            else:
                continue

            new_samples.append((path, new_label))
            new_targets.append(new_label)

        self.samples = new_samples
        self.targets = new_targets
        self.classes = [f"class_{i}" for i in range(21)]

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 64   # Ada 4000 can handle this
NUM_WORKERS = 8

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

train_dataset = CustomGroupedDataset("dataset/train", transform=train_transform)
val_dataset   = CustomGroupedDataset("dataset/val", transform=val_transform)
test_dataset  = CustomGroupedDataset("dataset/test", transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Total Classes:", 21)
print("Train size:", len(train_dataset))

Total Classes: 21
Train size: 21991


In [ ]:
model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=21
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)


scaler = torch.cuda.amp.GradScaler()

print("Model ready ✅")

Model ready ✅


/tmp/ipykernel_13055/1216949487.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.cuda.amp.autocast():
                outputs = model(images)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    cm = confusion_matrix(all_labels, all_preds)

    return acc, cm

In [ ]:
EPOCHS = 50
PATIENCE = 7

best_val_acc = 0.0
epochs_no_improve = 0

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0.0

    for images, labels in tqdm(train_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    # ---- Validate every 5 epochs ----
    if (epoch + 1) % 5 == 0:

        val_acc, cm = evaluate(model, val_loader)

        print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
        print(f"Train Loss: {epoch_loss:.4f}")
        print(f"Validation Accuracy: {val_acc:.4f}")
        print("Confusion Matrix:")
        print(cm)

        # ---- Check Improvement ----
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0

            torch.save({
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "best_val_acc": best_val_acc,
                "epoch": epoch + 1
            }, "best_flat_original.pt")

            print("✅ Saved new best model -> best.pt")

        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} validation steps")

        # ---- Early Stopping ----
        if epochs_no_improve >= PATIENCE:
            print("\n🛑 Early stopping triggered.")
            break

print("\n🔥 Training Finished")
print("Best Validation Accuracy:", best_val_acc)

In [ ]:
test_acc, test_cm = evaluate(model, test_loader)

print("\nFINAL TEST ACCURACY:", test_acc)

test_cm_df = pd.DataFrame(test_cm)
print("\nTest Confusion Matrix:")
print(test_cm_df)

In [ ]:
# Convert to dataframe if not already
test_cm_df = pd.DataFrame(test_cm)

print("\nTest Confusion Matrix:")
print(test_cm_df)

# ─── Per-Species Accuracy ───────────────────────────────────────────
species_total = test_cm_df.sum(axis=1)          # row sum
species_correct = np.diag(test_cm_df)           # diagonal
species_acc = species_correct / species_total

per_species_df = pd.DataFrame({
    "Species_ID": test_cm_df.index,
    "Correct": species_correct,
    "Total": species_total,
    "Accuracy": species_acc
})

print("\nPer-Species Accuracy:")
print(per_species_df)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    matthews_corrcoef,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)
from PIL import Image, ImageFile
import timm

ImageFile.LOAD_TRUNCATED_IMAGES = True

# ─── Config ─────────────────────────────────────────────────────────────────
DATASET_ROOT = "dataset"
CHECKPOINT   = "best_flat_2.pt"
IMAGE_SIZE   = 224
BATCH_SIZE   = 64
NUM_WORKERS  = 8
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")



def _build_class_names():
    """
    Returns a list indexed by new_label (0 … 18) whose values describe
    which original class-ID(s) the label corresponds to.
    """
    names = {}
    for cid in range(1, 4):          # 1, 2, 3
        names[cid - 1] = f"cls{cid}"
    for cid in range(5, 19):         # 5 … 18
        names[cid - 2] = f"cls{cid}"
    names[17] = "cls21"
    names[18] = "cls22-31"
    return [names[i] for i in range(19)]

ALL_CLASS_NAMES = _build_class_names()   # 19 names, labels 0-18


# ─── Dataset (same remapping as training) ────────────────────────────────────
class CustomGroupedDataset(ImageFolder):
    def __init__(self, root, transform=None):
        super().__init__(root, transform=transform)

        new_samples, new_targets = [], []
        for path, _ in self.samples:
            folder_name = path.split(os.sep)[-2]
            class_id = int(folder_name.split(".")[0])

            if 1 <= class_id <= 3:
                new_label = class_id - 1
            elif 5 <= class_id <= 18:
                new_label = class_id - 2
            elif class_id == 21:
                new_label = 17
            elif 22 <= class_id <= 31:
                new_label = 18
            else:
                continue

            new_samples.append((path, new_label))
            new_targets.append(new_label)

        self.samples = new_samples
        self.targets = new_targets

        self.classes = ALL_CLASS_NAMES


# ─── Evaluate ────────────────────────────────────────────────────────────────
def evaluate(model, loader, class_names, device):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast("cuda"):
                outputs = model(images)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)

    # ── Only use labels that actually appear in this split ──────────────────
    present_labels = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
    present_names  = [class_names[i] for i in present_labels]

    # ── Overall metrics ─────────────────────────────────────────────────────
    overall_acc  = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    kappa        = cohen_kappa_score(y_true, y_pred)
    mcc          = matthews_corrcoef(y_true, y_pred)

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", labels=present_labels, zero_division=0
    )
    p_weight, r_weight, f1_weight, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", labels=present_labels, zero_division=0
    )

    # ── Per-class metrics ────────────────────────────────────────────────────
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=present_labels, average=None, zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred, labels=present_labels)
    correct = np.diag(cm)
    total   = cm.sum(axis=1)
    per_class_acc = np.divide(
        correct, total,
        out=np.zeros_like(correct, dtype=float),
        where=(total != 0),
    )

    # ── Print per-species table ──────────────────────────────────────────────
    df = pd.DataFrame({
        "Species":   present_names,
        "Acc":       np.round(per_class_acc, 4),
        "Precision": np.round(precision,     4),
        "Recall":    np.round(recall,        4),
        "F1":        np.round(f1,            4),
        "Correct":   correct,
        "Total":     total,
    })

    print("\n" + "=" * 80)
    print("TEST RESULTS — per species")
    print("=" * 80)
    print(df.to_string(index=False))

    # ── Print aggregate metrics ──────────────────────────────────────────────
    print("\n" + "=" * 80)
    print("AGGREGATE METRICS")
    print("=" * 80)
    print(f"{'Metric':<30}{'Macro':>10}{'Weighted':>12}")
    print("-" * 52)
    print(f"{'Precision':<30}{p_macro:>10.4f}{p_weight:>12.4f}")
    print(f"{'Recall':<30}{r_macro:>10.4f}{r_weight:>12.4f}")
    print(f"{'F1 Score':<30}{f1_macro:>10.4f}{f1_weight:>12.4f}")
    print("-" * 52)
    print(f"{'Overall Accuracy':<30}{overall_acc:.4f}")
    print(f"{'Balanced Accuracy':<30}{balanced_acc:.4f}")
    print(f"{'Cohen Kappa':<30}{kappa:.4f}")
    print(f"{'Matthews Corr Coef (MCC)':<30}{mcc:.4f}")

    # ── Full sklearn report ──────────────────────────────────────────────────
    print("\n" + "=" * 80)
    print("FULL CLASSIFICATION REPORT")
    print("=" * 80)
    print(classification_report(
        y_true, y_pred,
        labels=present_labels,
        target_names=present_names,
        zero_division=0,
    ))

    # ── Confusion matrix heatmap ─────────────────────────────────────────────
    plt.figure(figsize=(14, 12))
    sns.heatmap(
        cm,
        cmap="Blues",
        xticklabels=present_names,
        yticklabels=present_names,
        annot=True,
        fmt="d",
        linewidths=0.5,
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150)
    print("\nConfusion matrix saved → confusion_matrix.png")

    # ── Per-species accuracy bar chart ───────────────────────────────────────
    plt.figure(figsize=(14, 6))
    bars = plt.bar(present_names, per_class_acc, color="steelblue")
    plt.axhline(overall_acc, color="red", linestyle="--", label=f"Overall acc ({overall_acc:.3f})")
    plt.xticks(rotation=90)
    plt.ylabel("Accuracy")
    plt.title("Per-Species Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig("per_species_accuracy.png", dpi=150)
    print("Per-species accuracy chart saved → per_species_accuracy.png")

    return overall_acc, cm


# ─── Main ────────────────────────────────────────────────────────────────────
def main():
    print(f"Using device: {DEVICE}")

    # ── Data ──────────────────────────────────────────────────────────────────
    val_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
    ])

    test_dataset = CustomGroupedDataset(
        os.path.join(DATASET_ROOT, "test"),
        transform=val_transform,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    # ── Model ─────────────────────────────────────────────────────────────────
    model = timm.create_model("efficientnet_b0", pretrained=False, num_classes=21)
    model = model.to(DEVICE)

    # ── Load BEST checkpoint (fix: was using last-epoch weights before) ────────
    if not os.path.exists(CHECKPOINT):
        raise FileNotFoundError(
            f"Checkpoint '{CHECKPOINT}' not found. "
            "Make sure best_flat_2.pt is in the working directory."
        )

    ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded checkpoint from epoch {ckpt.get('epoch', '?')} "
          f"(val_acc = {ckpt.get('best_val_acc', '?'):.4f})")

    # ── Evaluate ──────────────────────────────────────────────────────────────
    class_names = test_dataset.classes   # aligned with actual label mapping
    acc, cm = evaluate(model, test_loader, class_names, DEVICE)
    print(f"\nFINAL TEST ACCURACY: {acc:.4f}")


if __name__ == "__main__":
    main()